# Lab 5: extraction as a tool call

Lab 5 costs: $0.017

**One document in, one schema-valid record out, using a tool whose input schema is the
record you want back.**

Scenario 6, structured data extraction. Northwind's suppliers send invoices, receipts and
credit notes as whatever the sending system happened to produce: a letterhead, a till slip,
a scanned note with the reasons typed underneath. Downstream, a ledger wants the same handful
of fields off every one of them.

This lab builds that one step and nothing else. No vendor lookup, no validation loop, no
retry, no batch. Those are real parts of a real pipeline and the course teaches them; none of
them is what makes the extraction work.

**Two halves, in order.** First the mechanism: why a prompt asking for JSON is not a contract,
and what a forced tool call gives you instead. Then the schema design, which is where the
interesting failure lives: a field your document does not print, and what Claude does about it.

**This lab calls the API**, but every call is a single turn over a document of a dozen
lines, and a full run measures under two pence.

**It needs a key, and only a key.** It calls the Claude API directly rather than through
the Agent SDK, and a subscription authenticates the `claude` binary rather than the API. Put a
Console key in `ANTHROPIC_API_KEY` in the `.env` above this folder.

That difference is why this notebook imports `labkit.extract` by its full name rather than
taking helpers from `labkit`: it is a different world from `query()` and `ClaudeSDKClient`,
with no result message, no turns and no hooks.

## 1. Three documents to read

![Lab 5: extraction pipeline](../diagrams/lab-05-extraction-pipeline.png)


In [1]:
import json

import anthropic
import labkit
from labkit.extract import forced_call, show_record

lab = labkit.start(credential="api_key")

import documents

DOCS = documents.build(lab.workspace)
client = anthropic.Anthropic()

print()
print(DOCS["invoice"])

SystemExit: /Users/pizady/Code/ultimate-ccar-f-labs/.env has no value for ANTHROPIC_API_KEY. This lab calls the Claude API directly, so a subscription login will not carry it: it needs a Console key.

/Users/pizady/Code/ultimate-ccar-f-labs/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 2. The failure a schema prevents

**Start with the version everybody writes first: ask for JSON in the prompt and parse what
comes back.** It works often enough to reach production and fail there.

Watch what the next cell prints before it parses. What arrives is a good reply. It is just not
a record: there is a code fence around it, or a sentence introducing it, or an offer to break
the line items out separately. Any one of those, and `json.loads` raises on character one.

**The problem is not this particular reply.** It is that a prompt cannot promise anything
about the next one: not that it parses, not that it carries the same fields with the same
types, not that an absent value comes back empty rather than filled.

This cell deliberately does *not* use the helper the rest of the lab uses. The rawness is the
failure being demonstrated.

In [ ]:
asking_nicely = f"""Extract the vendor, the invoice number and the total from this
document. Reply with JSON only.

<document>
{DOCS["invoice"]}
</document>"""

response = client.messages.create(
    model=lab.model,
    max_tokens=1000,
    messages=[{"role": "user", "content": asking_nicely}],
)

text = next(block.text for block in response.content if block.type == "text")
print(f"stop_reason: {response.stop_reason}\n")
print(text)

try:
    print(f"\nparsed: {json.loads(text)}")
except json.JSONDecodeError as exc:
    print(f"\njson.loads failed: {exc}")

## 3. The extraction tool

**This is the mechanism the exam calls the most reliable one, and it starts with an
inversion worth saying out loud, because the rest follows from it.**

You define a tool, and what Claude calls that tool's **input** is what you are treating
as your **output**. There is no function named `extract_invoice` anywhere in this notebook. Nothing runs.

Three parts do the work:

| Part | Job |
|---|---|
| `input_schema` | The record you want back, written as the tool's arguments |
| `tool_choice` | Forces the call, so Claude cannot answer in prose instead |
| `tool_use.input` | Already a dict. Read the record straight off it |

**Two details inside `forced_call` are easy to skim past and both matter.** The block is found
by its **type**, not its position: a forced call usually puts it first, and *usually* is not a
contract. And the loop deliberately does not close. `stop_reason` comes back as `tool_use`,
which everywhere else in this course means send a `tool_result` back. Here you already have
what you wanted, so you stop.

In [ ]:
labkit.show_source(forced_call)

In [ ]:
extract_invoice = {
    "name": "extract_invoice",
    "description": "Extracts the header fields from a supplier invoice.",
    "input_schema": {
        "type": "object",
        "properties": {
            "vendor": {
                "type": "string",
                "description": "The supplier's name exactly as printed at the top.",
            },
            "invoice_number": {
                "type": "string",
                "description": "The supplier's own reference for this invoice.",
            },
            "stated_total": {
                "type": "number",
                "description": "The total the document states, as a number.",
            },
        },
        "required": ["vendor", "invoice_number", "stated_total"],
    },
}

block = forced_call(client, lab.model, DOCS["invoice"], extract_invoice)
show_record(block.input, label=f"tool called: {block.name}")

## 4. The model you validate is the schema you send

**A Pydantic model does both jobs from one definition.**

The schema above is a dict written by hand, and the record that came back is a dict too. So
nothing yet stops a typo in the schema, and nothing turns the reply into something your editor
understands.

`model_json_schema()` generates the schema you send, and `model_validate()` checks the record
you got and hands back a typed object. One source of truth, so the two cannot drift apart.

`Invoice` stays flat on purpose. A nested model would generate `$defs` and `$ref`, which is a
JSON Schema question rather than an extraction one; `line_items` is a list of strings so the
schema can show an array without it.

In [ ]:
from pydantic import BaseModel, Field


class Invoice(BaseModel):
    vendor: str = Field(description="The supplier's name exactly as printed at the top.")
    invoice_number: str = Field(description="The supplier's own reference for this invoice.")
    stated_total: float = Field(description="The total the document states, as a number.")
    line_items: list[str] = Field(description="One entry per description line charged for.")


extract_invoice = {
    "name": "extract_invoice",
    "description": "Extracts the header fields from a supplier invoice.",
    "input_schema": Invoice.model_json_schema(),
}

print("the schema the model is given:")
print(json.dumps(extract_invoice["input_schema"], indent=2))

block = forced_call(client, lab.model, DOCS["invoice"], extract_invoice)
record = Invoice.model_validate(block.input)
print(f"\ntyped record: {record!r}")
print(f"one field, as a float: {record.stated_total + 0}")

## 5. Designing for what the document does not print

**The receipt prints no purchase order and no date. Ask for both as required strings and
Claude has to put something in each field, so it does.**

Everything so far ran against the invoice, which prints every field the schema asked for.

Measured across twenty runs on the two models named in `.env.example`: every single one put
the sentinel `<UNKNOWN>` in the date, and most put it in the purchase order as well. Reword
the description slightly and a smaller model instead lifts the till receipt number into the
purchase order: **a real value, off the page, in the wrong field.**

Which of the two you get is a property of the model on the day. The shape of the defect is
not. Your ledger is handed a string, it type-checks, the key is required so it is always
present, and nothing downstream can tell it apart from a purchase order the supplier really
printed.

So the cell prints the record rather than asserting anything about it. What lands in those two
fields is Claude's to decide; this notebook's job is to show you that it decided.

In [ ]:
strict_receipt = {
    "name": "extract_receipt",
    "description": "Extracts the header fields from a supplier receipt.",
    "input_schema": {
        "type": "object",
        "properties": {
            "vendor": {"type": "string", "description": "The shop's name as printed."},
            "stated_total": {"type": "number", "description": "The total, as a number."},
            "purchase_order": {
                "type": "string",
                "description": "The purchase order this receipt is against.",
            },
            "issued_on": {
                "type": "string",
                "description": "The date printed on the receipt.",
            },
        },
        "required": ["vendor", "stated_total", "purchase_order", "issued_on"],
    },
}

block = forced_call(client, lab.model, DOCS["receipt"], strict_receipt)
show_record(block.input, label="required, and not nullable:")

**Nullable is about the type. Required is about the key. They are not alternatives.**

The next cell changes only the two types, keeps both keys required, and says in the
description what null means.

In [ ]:
nullable_receipt = json.loads(json.dumps(strict_receipt))   # copy, so the contrast survives
nullable_receipt["name"] = "extract_receipt_nullable"
properties = nullable_receipt["input_schema"]["properties"]
properties["purchase_order"] = {
    "type": ["string", "null"],
    "description": "The purchase order this receipt is against. "
                   "Null when the document does not print one.",
}
properties["issued_on"] = {
    "type": ["string", "null"],
    "description": "The date printed on the receipt. "
                   "Null when the document does not print one.",
}

block = forced_call(client, lab.model, DOCS["receipt"], nullable_receipt)
show_record(block.input, label="nullable, and still required:")

Both keys are still there, so the ledger always finds them. What changed is that **absence now
has a value it can be reported as.**

## 6. When the category is not on your list

**The last failure is the same one wearing different clothes.** A closed list of categories is
a schema demanding a string, so a reason outside the list gets forced into the nearest member,
confidently, and the record looks clean.

Two escape values fix it, and they are not the same escape:

| Value | Means | What you do about it |
|---|---|---|
| `unclear` | The source is ambiguous | Send it to a reviewer |
| `other` | The category is real, your list is short | Consider adding it to the list |

Both put the specifics in a companion detail field, so nothing is thrown away. The credit note
has one of each: a goodwill gesture, which is a real reason no sensible list carries, and a
line where the customer could not say what was wrong.

In [ ]:
extract_credit_note = {
    "name": "extract_credit_note",
    "description": "Extracts the credited lines from a supplier credit note.",
    "input_schema": {
        "type": "object",
        "properties": {
            "credit_note_number": {"type": "string", "description": "The note's reference."},
            "lines": {
                "type": "array",
                "description": "One entry per credited line.",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string", "description": "The goods credited."},
                        "amount": {"type": "number", "description": "The amount, as a number."},
                        "reason": {
                            "type": "string",
                            "enum": ["damaged", "wrong_item", "overcharge",
                                     "short_delivery", "unclear", "other"],
                            "description": "Use unclear when the note is ambiguous about "
                                           "why. Use other when it gives a real reason this "
                                           "list does not carry.",
                        },
                        "reason_detail": {
                            "type": ["string", "null"],
                            "description": "The reason as printed. Required when reason is "
                                           "unclear or other. Null otherwise.",
                        },
                    },
                    "required": ["description", "amount", "reason", "reason_detail"],
                },
            },
        },
        "required": ["credit_note_number", "lines"],
    },
}

block = forced_call(client, lab.model, DOCS["credit_note"], extract_credit_note)

print(f"credit note {block.input['credit_note_number']}")
labkit.show_table([(f"{line['amount']:.2f}", line["reason"], line["reason_detail"])
                   for line in block.input["lines"]],
                  headers=("amount", "reason", "reason_detail"),
                  wrap={"reason_detail": 52})

**Neither line was forced into `damaged` or `wrong_item`, and neither reason was lost.**

## What you built

| Diagram box | Where it was built |
|---|---|
| Supplier documents | Section 1 |
| `extract_metadata`, forced tool choice | Sections 3 and 4 |
| JSON Schema | Sections 3, 5 and 6 |
| Pydantic validation | Section 4 |

The diagram carries more boxes than this lab does, on purpose. Vendor lookup through a tool
loop, semantic validation, the repair loop, field-level confidence and review routing, and the
whole Message Batches lane are taught in Chapters 16, 17 and 19 and are not practised here.
**This lab is the step they all sit on top of.**

**The decision to carry out of it:** given a field your downstream system always reads and a
document that may not print it, make the type nullable and keep the key required. Nullable is
about the type, required is about the key, and they are not alternatives. That pairing is what
turns an absence into a value your code can act on instead of a plausible invention nobody
queries.